In [1]:
import pandas as pd
from classify import Classifier
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns

/home/lam/miniforge3/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


In [5]:
clf = Classifier("cfg/car_quality.yaml")  # load task config
ckpt_path = (r"output/20260729/CarQuality.Classification/mobilenet_v3_small"
             r"/version_1/checkpoints/best_acc_epoch=97-val_loss=0.047-val_acc=0.992.ckpt")
results = clf.predict(
    source="datasets/dataset",
    ckpt_path=ckpt_path,
    num_classes=2,
    precision="32",
    resize=128,
)

df = pd.DataFrame(results)
df.to_csv("results.csv", index=False)


2026-08-07 13:25:44 | INFO     | classify — Classifier ready — model=mobilenet_v3_small  classes=0
2026-08-07 13:25:44 | INFO     | classify.engine.predictor — Loading model 'mobilenet_v3_small' from 'output/20260729/CarQuality.Classification/mobilenet_v3_small/version_1/checkpoints/best_acc_epoch=97-val_loss=0.047-val_acc=0.992.ckpt' on cuda…
2026-08-07 13:25:44 | INFO     | classify.models.lightning_module — Loaded pretrained weights from 'output/20260729/CarQuality.Classification/mobilenet_v3_small/version_1/checkpoints/best_acc_epoch=97-val_loss=0.047-val_acc=0.992.ckpt'.
2026-08-07 13:25:45 | INFO     | classify.engine.predictor —   0028e49b-4e02-4ed1-b9c6-550f43e08a78_aligned.jpg  →  class=0  score=0.9998
2026-08-07 13:25:45 | INFO     | classify.engine.predictor —   00dd301c-c6c7-4fca-b7c6-30311a9c816c_aligned.jpg  →  class=0  score=1.0000
2026-08-07 13:25:45 | INFO     | classify.engine.predictor —   0107d7c0-d826-40a7-8a5f-3fd14181b28f.jpg  →  class=0  score=0.9987
2026-08-07 

In [6]:
df

,file,class_index,score
0,datasets/dataset/train/0028e49b-4e02-4ed1-b9c6...,0,0.999795
1,datasets/dataset/train/00dd301c-c6c7-4fca-b7c6...,0,0.999957
2,datasets/dataset/train/0107d7c0-d826-40a7-8a5f...,0,0.998721
3,datasets/dataset/train/015a6379-065e-4ce3-a2e5...,0,0.954086
4,datasets/dataset/train/01add812-5843-4f75-a2fa...,0,0.981897
...,...,...,...
3494,datasets/dataset/train/z782380696570_07ed9dea7...,1,0.999978
3495,datasets/dataset/train/z786023933745_09a681ea9...,0,0.997646
3496,datasets/dataset/train/z792241892678_8859f0f19...,0,0.998258
3497,datasets/dataset/train/z793142113256_d34482ab9...,1,0.999843


In [8]:
labels = ["colored", "photo"]


df["predicted_label"] = df["class_index"].apply(lambda x: labels[x])
df

,file,class_index,score,predicted_label
0,datasets/dataset/train/0028e49b-4e02-4ed1-b9c6...,0,0.999795,colored
1,datasets/dataset/train/00dd301c-c6c7-4fca-b7c6...,0,0.999957,colored
2,datasets/dataset/train/0107d7c0-d826-40a7-8a5f...,0,0.998721,colored
3,datasets/dataset/train/015a6379-065e-4ce3-a2e5...,0,0.954086,colored
4,datasets/dataset/train/01add812-5843-4f75-a2fa...,0,0.981897,colored
...,...,...,...,...
3494,datasets/dataset/train/z782380696570_07ed9dea7...,1,0.999978,photo
3495,datasets/dataset/train/z786023933745_09a681ea9...,0,0.997646,colored
3496,datasets/dataset/train/z792241892678_8859f0f19...,0,0.998258,colored
3497,datasets/dataset/train/z793142113256_d34482ab9...,1,0.999843,photo


In [9]:
filenames = df["file"]
labels = df["predicted_label"]

In [10]:
import os
import shutil
def copy_files_to_folder(paths, labels, dest_folder):
    for filename, label in zip(paths, labels):
        label_folder = os.path.join(dest_folder, label)
        os.makedirs(label_folder, exist_ok=True)
        dest_path = os.path.join(label_folder, os.path.basename(filename))
        shutil.copy(filename, dest_path)

In [11]:
copy_files_to_folder(df["file"], df["predicted_label"], "predicted_results")

In [24]:
import glob
from sklearn.model_selection import train_test_split

input_dir = "predicted_results"
output_folder = "dataset_4"

for label in os.listdir(input_dir):
    label_folder = os.path.join(input_dir, label)
    if os.path.isdir(label_folder):
        image_paths = glob.glob(os.path.join(label_folder, "*"))
        train_paths, val_paths = train_test_split(image_paths, test_size=0.2, random_state=42)

        # Create output directories for train and validation sets
        train_output_dir = os.path.join(output_folder, "train", label)
        val_output_dir = os.path.join(output_folder, "val", label)
        os.makedirs(train_output_dir, exist_ok=True)
        os.makedirs(val_output_dir, exist_ok=True)\section{Generalization to Real-World and Telephony Conditions}


        # Copy training images
        for path in train_paths:
            shutil.copy(path, os.path.join(train_output_dir, os.path.basename(path)))

        # Copy validation images
        for path in val_paths:
            shutil.copy(path, os.path.join(val_output_dir, os.path.basename(path)))


In [26]:
import pathlib
import json

folder = r"predicted_results/photo"
image_paths = os.listdir(folder)  # Adjust the pattern if needed

json_dump = dict()
json_file = []

for image_path in image_paths:
    json_file.append({
        "image_path": image_path,
        "label": ["photo"]
    })
print(json_file)

with open("tags.json", "w") as f:
    json.dump(json_file, f, indent=4)

[{'image_path': '1 (8) dated_0.jpg', 'label': ['photo']}, {'image_path': '1 (8) dated_1.jpg', 'label': ['photo']}, {'image_path': '1 TONG THE  (8).jpg', 'label': ['photo']}, {'image_path': '1 TONG THE  (8)_0.jpg', 'label': ['photo']}, {'image_path': '1 TONG THE  (8)_1.jpg', 'label': ['photo']}, {'image_path': '1468486733824_11240_0.jpg', 'label': ['photo']}, {'image_path': '1469618546810_15097_1.jpg', 'label': ['photo']}, {'image_path': '1470039818684_15971_1.jpg', 'label': ['photo']}, {'image_path': '1470210967795_16203_0.jpg', 'label': ['photo']}, {'image_path': '1473152626281_23391.jpg', 'label': ['photo']}, {'image_path': '1473843077455_26330_1.jpg', 'label': ['photo']}, {'image_path': '1476197219764_32888_0.jpg', 'label': ['photo']}, {'image_path': '1476267278670_33132_1.jpg', 'label': ['photo']}, {'image_path': '1476340747997_33524_0.jpg', 'label': ['photo']}, {'image_path': '1476958922728_34948_0.jpg', 'label': ['photo']}, {'image_path': '1477105354275_35279_1.jpg', 'label': ['p